# 推理引擎源码与前沿优化 · 第 2/8 课：vLLM V1 统一调度器、KVCacheManager 与 Prefix Cache

> 状态：**未开始**  
> 源码审阅日期：2026-08-12；vLLM `8e958902eee5`；SGLang `9deb6952afa4`。

## 本课目标与通过标准

本课产出：能逐段解释 `Scheduler.schedule` 如何用统一 token budget 覆盖 prefill/decode/speculation，并追踪 prefix hit、block 分配、抢占和释放的完整状态变化。

通过要求：唯一代码填空题通过全部断言；Q1～Q3 都能沿源码对象给出因果链；能指出一个正确性不变量、一个性能边界和一个需要 benchmark 才能确认的结论；总分至少 8/10。

## 源码版本与阅读方法

本课程使用固定 commit 的永久链接保证行号可复现，同时链接 current docs 供核对最新变化。阅读时先画调用图和状态所有权，再进入分支；不要从大文件第一行机械顺读。

源码阅读主轴：

1. `vllm/v1/core/sched/scheduler.py::schedule`：先读顶部注释，再标记 RUNNING 循环、WAITING 接纳、`allocate_slots` 失败与 `_preempt_request`。
2. `vllm/v1/core/kv_cache_manager.py::get_computed_blocks`：理解为何只复用完整 block、全命中仍要重算最后 token。
3. 同文件 `allocate_slots`：区分 cached blocks、new computed blocks、lookahead slots 与 watermark/reservation。
4. `vllm/v1/core/block_pool.py`：追 `get_cached_block`、`get_new_blocks`、`cache_full_blocks` 和 free queue。
5. `docs/design/prefix_caching.md` 与 `hybrid_kv_cache_manager.md`：把源码细节还原为设计约束。

源码快照：vLLM `8e958902eee5`。

## 核心对象

V1 scheduler 不把请求硬分成 prefill batch 和 decode batch。每个请求只有 `num_computed_tokens` 与 `num_tokens_with_spec`；本轮缺口就是待计算 token。固定的全局 token budget 被依次分给请求，因此 chunked prefill、普通 decode、prefix hit 后续算和 speculative verification 共享一套调度表达。

KVCacheManager 是 scheduler 与 block pool/多 KV group coordinator 的接口。Scheduler 决定“本轮算多少”，KV manager 决定“这些 token 是否有可用 block、哪些前缀已算过”。

## 调用链与状态变化

新请求进入 WAITING 后，scheduler 先做本地/外部 prefix lookup，得到完整 computed blocks 和命中 token 数；再依据本轮 token 数调用 `allocate_slots`。若容量不足，运行中最低优先级/队尾请求可能被 preempt，释放其 blocks 并重置计算进度。成功后 `SchedulerOutput` 同时携带每请求 token 数与 block ids，worker 据此写 KV。

执行完成后 `update_from_output` 增加 computed tokens、接纳采样/投机结果、检测停止条件；完成请求由 `_free_request` 释放或把完整 block 进入 prefix cache。逻辑计数和物理 block 生命周期必须一起变化。

## 正确性条件与常见误区

不变量：`0 ≤ num_computed_tokens ≤ num_tokens_with_spec`；只有完整、哈希一致且 namespace/模型配置兼容的 block 可复用；prefix hit 不能越过需要 logits 的最后位置；block 不能同时处于 free queue 和被请求引用；抢占后不能保留旧 spec tokens 当作已验证输出。

`scheduler_reserve_full_isl` 与 watermark 是防止“首 chunk 能放下、完整 prompt 放不下”导致反复抢占的接纳控制，不是提高理论显存容量。

## 当前前沿与工程取舍

Prefix cache 命中降低 prefill 计算，但不减少权重常驻量；block 越小复用/碎片粒度更细，metadata 与 hash 成本更高。积极接纳提高瞬时 batch，却可能触发 preemption/recompute，使 goodput 和 p99 一起下降。最新源码还需处理 hybrid KV layout、Mamba/SWA、KV connector 和稀疏保留组，因此不能用旧版单一 block table 心智模型替代当前 coordinator。

## 具体推演

请求 A：`computed=6,total=10`，请求 B：`computed=0,total=7`，全局 budget=8。统一算法先给 A 4 个 token，再给 B 4 个；它不需要知道 A 是 decode/spec verify、B 是 chunked prefill。若 A 的 4 个 token 需要新 block 且分配失败，scheduler 必须先选择抢占或停止，而不是只修改数字预算。

请先口头复述“输入 → 状态所有者 → 状态迁移 → 输出/指标”，再做练习。

## 实践任务：唯一代码填空题

实现统一 token budget 的教学版计划器。RUNNING 请求优先于 WAITING；每个请求按 `total_with_spec-computed` 领取预算，不引入 prefill/decode 分支。

只能修改 `TODO`/`______` 位置，不得删除断言或放宽通过条件。

In [ ]:
def unified_token_plan(requests, token_budget):
    if token_budget < 0:
        raise ValueError("negative token budget")
    if any(r["computed"] < 0 or r["total_with_spec"] < r["computed"] for r in requests):
        raise ValueError("invalid request counters")

    # Scheduler 源码先处理 running，再处理 waiting；同组保持输入次序。
    ordered = [r for r in requests if r["status"] == "running"] + [
        r for r in requests if r["status"] == "waiting"
    ]
    plan = {}
    for request in ordered:
        if token_budget == 0:
            break
        # TODO：只根据统一 token 缺口分配，不判断阶段名称。
        need = ______
        take = ______
        if take:
            plan[request["id"]] = take
            token_budget -= take
    return plan, token_budget

reqs = [
    {"id": "prefill", "status": "waiting", "computed": 0, "total_with_spec": 7},
    {"id": "decode_or_spec", "status": "running", "computed": 6, "total_with_spec": 10},
]
assert unified_token_plan(reqs, 8) == ({"decode_or_spec": 4, "prefill": 4}, 0)
assert unified_token_plan(reqs, 20) == ({"decode_or_spec": 4, "prefill": 7}, 9)


### 检查方法

运行断言；增加一个 `computed==total_with_spec` 的请求，确认它不消耗预算，再解释真实源码还需通过哪些 KV/encoder 条件。

### Q1

为什么统一 token budget 能同时表达 chunked prefill、decode 与 speculative verification？它没有消除什么复杂度？

**你的答案：**


### Q2

Prefix cache 全命中 prompt 时，为什么源码仍限制最大命中到 `prompt_length-1`？

**你的答案：**


### Q3

`allocate_slots` 失败后立即把 `max_num_seqs` 调大是否合理？请给出源码级因果链。

**你的答案：**


## 评分规则

- 代码 4 分：主路径 2 分、边界条件 1 分、能映射回源码对象 1 分；
- Q1～Q3 各 2 分：必须包含对象、状态变化、正确性或成本链；
- 一票否决：把论文峰值写成普遍生产结论；把源码支持写成所有模型/硬件可用；混淆算法正确性与性能；只背类名而说不清状态所有权。

## 参考资料

- [vLLM V1 Scheduler source](https://github.com/vllm-project/vllm/blob/8e958902eee56ca5158728f1dd5a32246f0246f3/vllm/v1/core/sched/scheduler.py#L439-L1264)
- [vLLM KVCacheManager source](https://github.com/vllm-project/vllm/blob/8e958902eee56ca5158728f1dd5a32246f0246f3/vllm/v1/core/kv_cache_manager.py#L230-L628)
- [vLLM BlockPool source](https://github.com/vllm-project/vllm/blob/8e958902eee56ca5158728f1dd5a32246f0246f3/vllm/v1/core/block_pool.py)
- [vLLM prefix caching design](https://docs.vllm.ai/en/latest/design/prefix_caching/)
- [vLLM hybrid KV cache manager design](https://github.com/vllm-project/vllm/blob/8e958902eee56ca5158728f1dd5a32246f0246f3/docs/design/hybrid_kv_cache_manager.md)
- [vLLM scheduler configuration](https://docs.vllm.ai/en/latest/api/vllm/config/scheduler/)

源码链接固定到本课审阅 commit；current docs、支持矩阵和默认参数会变化，面试或部署前必须按目标版本重新核对。